<a href="https://colab.research.google.com/github/zaldyzufar/Tugas_AI/blob/main/dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import numpy as np
import pandas as pd

def load_and_preprocess(file_path, sheet_name=None):
    if file_path.endswith('.csv'):
        df = pd.read_csv(file_path)
    elif file_path.endswith('.xlsx'):
        df = pd.read_excel(file_path, sheet_name=sheet_name)
    else:
        raise ValueError("Unsupported file type. Please provide a .csv or .xlsx file.")

    data = df[['sexo', 'edad_cat', 'region_residencia']].dropna()

    data['sexo'] = data['sexo'].map({'hombre': 1, 'mujer': 0})

    features = pd.get_dummies(data[['edad_cat', 'region_residencia']])


    X = features.values.T.astype(float)
    Y = data['sexo'].values.reshape(1, -1).astype(float)

    return X, Y

def initialize_parameters(input_size, hidden_size, output_size):
    np.random.seed(42)
    parameters = {
        "W1": np.random.randn(hidden_size, input_size) * 0.01,
        "b1": np.zeros((hidden_size, 1)),
        "W2": np.random.randn(output_size, hidden_size) * 0.01,
        "b2": np.zeros((output_size, 1))
    }
    return parameters

def sigmoid(Z):
    return 1 / (1 + np.exp(-Z))

def relu(Z):
    return np.maximum(0, Z)

def relu_derivative(Z):
    return (Z > 0).astype(int)

def forward_propagation(X, parameters):
    W1, b1, W2, b2 = parameters["W1"], parameters["b1"], parameters["W2"], parameters["b2"]

    Z1 = np.dot(W1, X) + b1
    A1 = relu(Z1)
    Z2 = np.dot(W2, A1) + b2
    A2 = sigmoid(Z2)

    cache = {"Z1": Z1, "A1": A1, "Z2": Z2, "A2": A2}
    return A2, cache

def compute_cost(Y, A2):
    m = Y.shape[1]
    # Menambahkan epsilon kecil untuk menghindari log(0)
    epsilon = 1e-15
    A2 = np.clip(A2, epsilon, 1 - epsilon)
    cost = -np.sum(Y * np.log(A2) + (1 - Y) * np.log(1 - A2)) / m
    return np.squeeze(cost)

def backward_propagation(X, Y, parameters, cache):
    m = X.shape[1]
    W2 = parameters["W2"]

    dZ2 = cache["A2"] - Y
    dW2 = np.dot(dZ2, cache["A1"].T) / m
    db2 = np.sum(dZ2, axis=1, keepdims=True) / m

    dZ1 = np.dot(W2.T, dZ2) * relu_derivative(cache["Z1"])
    dW1 = np.dot(dZ1, X.T) / m
    db1 = np.sum(dZ1, axis=1, keepdims=True) / m

    grads = {"dW1": dW1, "db1": db1, "dW2": dW2, "db2": db2}
    return grads

def update_parameters(parameters, grads, learning_rate):
    for key in parameters.keys():
        parameters[key] -= learning_rate * grads["d" + key]
    return parameters

def train_neural_network(X, Y, hidden_size=16, epochs=5000, learning_rate=0.1):
    input_size = X.shape[0]
    output_size = Y.shape[0]

    parameters = initialize_parameters(input_size, hidden_size, output_size)

    print(f"Memulai pelatihan dengan {X.shape[1]} data...")
    for i in range(epochs):
        A2, cache = forward_propagation(X, parameters)
        cost = compute_cost(Y, A2)
        grads = backward_propagation(X, Y, parameters, cache)
        parameters = update_parameters(parameters, grads, learning_rate)

        if i % 500 == 0:
            print(f"Epoch {i}: Cost = {cost:.6f}")

    return parameters

def predict(X, parameters):
    A2, _ = forward_propagation(X, parameters)
    return (A2 > 0.5).astype(int)

# Pastikan nama file sesuai dengan file yang Anda unggah
file_name = 'Hantavirus_chile.xlsx'
X_train, Y_train = load_and_preprocess(file_name, sheet_name='Sheet1')

# Melatih model
trained_parameters = train_neural_network(
    X_train, Y_train, hidden_size=16, epochs=5000, learning_rate=0.1
)
# Evaluasi Akurasi
predictions = predict(X_train, trained_parameters)
accuracy = np.mean(predictions == Y_train) * 100
print(f"\nAkurasi Akhir: {accuracy:.2f}%")

Memulai pelatihan dengan 1293 data...
Epoch 0: Cost = 0.693220
Epoch 500: Cost = 0.596506
Epoch 1000: Cost = 0.596383
Epoch 1500: Cost = 0.595858
Epoch 2000: Cost = 0.593909
Epoch 2500: Cost = 0.589685
Epoch 3000: Cost = 0.585930
Epoch 3500: Cost = 0.583570
Epoch 4000: Cost = 0.581597
Epoch 4500: Cost = 0.579741

Akurasi Akhir: 71.77%
